In [27]:
import pandas as pd
from selenium import webdriver
from selenium.webdriver.common.keys import Keys
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from selenium.common.exceptions import NoSuchElementException, TimeoutException, NoSuchAttributeException
import time
import re
import collections
import csv
import requests
import random


pd.set_option('display.max_columns', None, 'display.max_rows', None)

In [10]:
driver = webdriver.Chrome()
driver.get('https://www.linkedin.com/login?fromSignIn=true&trk=guest_homepage-basic_nav-header-signin')

username_field = driver.find_element(By.NAME, 'session_key')
password_field = driver.find_element(By.NAME, 'session_password')
username_field.send_keys('alejandro.shutov@gmail.com')
password_field.send_keys('Alejo26@lejo26')
password_field.send_keys(Keys.RETURN)


print("Press 0 and Enter to continue...")
while True:
    time.sleep(10)
    print('waiting')
    user_input = input()
    if user_input == '0':
        break


driver.get('https://www.linkedin.com/my-items/saved-jobs/')

url_list = []
while True:
    try:
        WebDriverWait(driver, 10).until(EC.element_to_be_clickable((By.CLASS_NAME, 'scale-down')))
        page = driver.find_element(By.CLASS_NAME, 'workflow-results-container')
        time.sleep(1)
        jobs = page.find_elements(By.CLASS_NAME, 'entity-result__title-text')
        urls = [(job.find_element(By.CLASS_NAME, 'app-aware-link').get_attribute('href'), job.find_element(By.CLASS_NAME, 'app-aware-link').text) for job in jobs]
        url_list += urls
        WebDriverWait(driver, 10).until(EC.element_to_be_clickable((By.CLASS_NAME, 'artdeco-pagination__button--next')))
        page.find_element(By.CLASS_NAME, 'artdeco-pagination__button--next').click()
        time.sleep(2)
    except NoSuchElementException:
        print('no such element')
        break
    except TimeoutException:
        print('timeout')
        break

Press 0 and Enter to continue...
waiting
10
20
27
35
42
50
58
66
75
83
92
99
106
116
122
131
141
151
158
168
176
185
195
205
213
223
233
243
253
263
269
timeout


In [22]:
# with open('data/output.csv', 'w', newline='', encoding='utf-8') as file:
#     writer = csv.writer(file)
#     writer.writerows(url_list)

In [68]:
# job_data[-1]
len(job_data)/len(url_list)

0.9968944099378882

In [64]:
driver = webdriver.Chrome()
driver.get('https://www.linkedin.com/login?fromSignIn=true&trk=guest_homepage-basic_nav-header-signin')

print("Press 0 and Enter to continue...")
while True:
    time.sleep(5)
    user_input = input()
    if user_input:
        break

if job_data:
    pass
else:
    job_data = []

for i, url in enumerate(url_list[(len(job_data)):]):
    try:
        driver.get(url[0])
        time.sleep(random.uniform(1, 2))

        WebDriverWait(driver, 5).until(EC.element_to_be_clickable((By.CLASS_NAME, 'job-details-jobs-unified-top-card__primary-description-container')))
        page = driver.find_element(By.CLASS_NAME, 'scaffold-layout__main')

        driver.execute_script("arguments[0].scrollIntoView(true);", page.find_element(By.CLASS_NAME, 'jobs-description-content__text'))
        time.sleep(random.uniform(0.5, 1.5)) 

        title = page.find_element(By.CLASS_NAME, 'job-details-jobs-unified-top-card__job-title').text
        subtitle = page.find_element(By.CLASS_NAME, 'job-details-jobs-unified-top-card__primary-description-container')
        details = subtitle.find_elements(By.CLASS_NAME, 'tvm__text')
        details = [element.text for element in details]

    except (TimeoutException, NoSuchElementException):
        continue
    except Exception as e:
        print(f"Error processing URL {url}: {e}")
        continue

    try:
        WebDriverWait(driver, 2).until(EC.element_to_be_clickable((By.CLASS_NAME, 'jobs-description__footer-button')))
        page.find_element(By.CLASS_NAME, 'jobs-description__footer-button').click()
    except (TimeoutException, NoSuchElementException):
        pass

    description = page.find_element(By.CLASS_NAME, 'jobs-description-content__text').find_element(By.CLASS_NAME, 'mt4').text

    try:
        skills = page.find_element(By.CLASS_NAME, 'job-details-how-you-match__skills-section-descriptive-skill').text
    except NoSuchElementException:
        skills = page.find_elements(By.CLASS_NAME, 'job-details-how-you-match__skills-item-subtitle')
        skills = [skill.text for skill in skills]

    job_data.append([title, details, description, skills])

pd.DataFrame(job_data, columns=['title', 'details', 'description', 'skills']).to_csv('data/linkedin_jobdata.csv', index=False)

Press 0 and Enter to continue...


In [89]:
df = pd.read_csv('data/linkedin_jobdata.csv')

In [90]:
def safe_eval(cell):
    try:
        return eval(cell)
    except Exception:
        return cell

df.skills = df.skills.apply(safe_eval)

In [91]:
def correct_list(cell):
    if isinstance(cell, list):
        skills = []
        for i in cell:
            skills += [skill.strip() for skill in re.split(',| and ', i)]
        skills = [skill[4:] if skill.startswith('and') else skill for skill in skills]
        skills = [skill for skill in skills if skill != '']
        return skills
    else:
        return cell

df.skills = df.skills.apply(correct_list)

In [92]:
def convert_to_list(cell):
    if isinstance(cell, str):
        # Split the string at '·' character
        skills = [skill.strip() for skill in cell.split('·')]
        # Further split each skill at ',' character
        skills = [subskill.strip() for skill in skills for subskill in skill.split(',')]
        # Remove 'and' if it appears at the end of a skill
        skills = [skill[:-4] if skill.endswith(' and') else skill for skill in skills]
        return skills
    else:
        return cell

df.skills = df.skills.apply(convert_to_list)


In [117]:
skills_count = collections.Counter(skill for skills_list in df['skills'] for skill in skills_list)

In [118]:
skills_list = []
with open('linkedin_skills.txt', 'r') as file:
    lines = file.readlines()
    for line in lines:
        half_length = len(line.strip()) // 2
        line = line[:half_length]
        skills_list.append(line)

skills_dict = {}
for skill in skills_list:
    count = skills_count.get(skill, 0)
    skills_dict[skill] = count
sorted(skills_dict.items(), key= lambda x:x[1], reverse=True)

[('User Experience (UX)', 467),
 ('User Interface Design', 391),
 ('User Experience Design (UED)', 331),
 ('English', 257),
 ('Figma (Software)', 253),
 ('Communication', 238),
 ('Wireframing', 233),
 ('Design', 194),
 ('Usability', 152),
 ('Cascading Style Sheets (CSS)', 141),
 ('Product Design', 128),
 ('User Research', 113),
 ('HTML', 101),
 ('Analytical Skills', 101),
 ('Usability Testing', 98),
 ('UX Research', 93),
 ('Interaction Design', 92),
 ('Design Thinking', 89),
 ('JavaScript', 87),
 ('Prototyping', 76),
 ('Information Architecture', 73),
 ('Problem Solving', 73),
 ('Product Management', 73),
 ('Human Computer Interaction', 72),
 ('User-centered Design', 69),
 ('Teamwork', 68),
 ('Visual Design', 66),
 ('Project Management', 65),
 ('User Flows', 65),
 ('Adobe Photoshop', 64),
 ('Web Design', 63),
 ('Adobe Illustrator', 59),
 ('Graphic Design', 58),
 ('Design Systems', 55),
 ('Marketing', 54),
 ('Agile Methodologies', 53),
 ('Research Skills', 48),
 ('User Stories', 44),
 (

In [114]:
for key in skills_list:
    del skills_count[key]
skills_count

Counter({'Skill Development': 20,
         'Customer Experience': 17,
         'Time Management': 16,
         'Storyboarding': 16,
         'Digital Designs': 16,
         'Databases': 16,
         'Presentation Skills': 16,
         'Oral Communication': 15,
         'Java': 15,
         'Technical Documentation': 14,
         'WordPress': 14,
         'Business Requirements': 14,
         'Team Leadership': 14,
         'Negotiation': 14,
         'Microsoft PowerPoint': 14,
         'Engineering': 14,
         'Reporting': 14,
         'Hyper-Converged Infrastructure': 14,
         'B2C': 13,
         'Business-to-Business (B2B)': 13,
         'Confluence': 13,
         'Product Strategy': 13,
         'Content Management Systems (CMS)': 13,
         'Web Analytics': 13,
         'Git': 13,
         'German': 13,
         'Front-End Design': 13,
         'Consulting': 12,
         'Sales': 12,
         'Product Requirements': 12,
         'Salesforce.com': 12,
         'Zeplin': 12

In [115]:
skills_list

['Process Optimization',
 'Process Modeling',
 'Python',
 'Supply Chain Optimization',
 'SQL',
 'Data Analysis',
 'Project Management',
 'Data Analytics',
 'Research Skills',
 'UX Research',
 'User Research',
 'User Stories',
 'User Personas',
 'User Journeys',
 'User Flows',
 'User Experience (UX)',
 'User Experience Design (UED)',
 'User-centered Design',
 'Experience Design',
 'Interaction Design',
 'Human Computer Interaction',
 'Information Architecture',
 'Design Thinking',
 'Design Standards',
 'Design Systems',
 'Design Tools',
 'Design',
 'Product Design',
 'Service Design',
 'Wireframing',
 'Prototyping',
 'Mockups',
 'Figma (Software)',
 'Sketch App',
 'Axure RP',
 'InVision',
 'Adobe Creative Suite',
 'Adobe XD',
 'Adobe InDesign',
 'Adobe Photoshop',
 'Adobe Illustrator',
 'After Effects',
 'Visual Design',
 'Graphic Design',
 'Graphic Design Principles',
 'Typography',
 'Interfaces',
 'User Interface Design',
 'HTML',
 'HTML5',
 'Cascading Style Sheets (CSS)',
 'JavaScrip

In [86]:
def categorize_title(title):
    title = title.lower()
    if 'analyst' in title or 'analista' in title or 'cro' in title:
        return 'Analyst'
    elif 'designer' in title or 'ux' in title or 'ui' in title or 'maquetador' in title or 'diseñador' in title:
        return 'Designer'
    elif 'developer' in title or 'programador' in title:
        return 'Developer'
    elif 'consultant' in title or 'strategist' in title:
        return 'Consultant'
    elif 'product owner' in title or 'product manager' in title:
        return 'Product Management'
    elif 'crm' in title or 'customer success' in title:
        return 'Customer Relationship'
    elif 'research' in title:
        return 'Research'
    else:
        return 'Other'

df['title_category'] = df['title'].apply(categorize_title)

In [87]:
df['skills_str'] = df['skills'].apply(lambda x: str(x)[1:-1])
df['concatenated'] = df['title'].astype(str) + ' ' + df['description'].astype(str) + ' ' + df['skills_str'].astype(str)

# Create one massive string
massive_string = ' '.join(df['concatenated'])
skills_str = df['skills_str'].astype(str)

In [88]:
df

,title,details,description,skills,title_category,skills_str,concatenated
0,UX/UI Designer,['1 year ago'],SUMMARY \nAs a UX/UI Designer you will play a ...,"[Design, User Experience (UX), User Interface ...",Designer,"'Design', 'User Experience (UX)', 'User Interf...",UX/UI Designer SUMMARY \nAs a UX/UI Designer y...
1,Webflow Developer,['1 year ago'],We are looking for an experienced Webflow Desi...,"[Cascading Style Sheets (CSS), Design, User Ex...",Developer,"'Cascading Style Sheets (CSS)', 'Design', 'Use...",Webflow Developer We are looking for an experi...
2,Especialista en Marketing Digital y Diseño UX/UI,['1 year ago'],En QOQOBO queremos conectar el mundo de la cos...,"[Adobe Creative Cloud, Adobe Photoshop, Market...",Designer,"'Adobe Creative Cloud', 'Adobe Photoshop', 'Ma...",Especialista en Marketing Digital y Diseño UX/...
3,Senior Product Design (m/f/d),['1 year ago'],Hello. We are BestSecret. And we will tell you...,[],Other,,Senior Product Design (m/f/d) Hello. We are Be...
4,Product Designer,['Reposted 1 year ago'],"En idealista, portal inmobiliario líder en Esp...","[Design, Product Management, User Personas, B2...",Designer,"'Design', 'Product Management', 'User Personas...","Product Designer En idealista, portal inmobili..."
5,Product Specialist,['Reposted 1 year ago'],"¿Estas buscando un cambio? \n\n\nSomos Vortal,...","[English, Computer Engineering, Documentation,...",Other,"'English', 'Computer Engineering', 'Documentat...",Product Specialist ¿Estas buscando un cambio? ...
6,IT Product Manager (H/M),['1 year ago'],Clikalia es la Proptech Nº #1 en la digitaliza...,"[Communication, Problem Solving, Product Devel...",Product Management,"'Communication', 'Problem Solving', 'Product D...",IT Product Manager (H/M) Clikalia es la Propte...
7,Experience Designer,['1 year ago'],Experience Designer \nSobre ti:\n\nSi eres esa...,"[Design, Design Thinking, User Personas, B2C, ...",Designer,"'Design', 'Design Thinking', 'User Personas', ...",Experience Designer Experience Designer \nSobr...
8,Associate Product Designer,['1 year ago'],ASSOCIATE DIGITAL PRODUCT DESIGNER\n\nExperien...,[],Designer,,Associate Product Designer ASSOCIATE DIGITAL P...
9,Product Designer,['Reposted 1 year ago'],DIGITAL PRODUCT DESIGNER\n\nExperience Design\...,[],Designer,,Product Designer DIGITAL PRODUCT DESIGNER\n\nE...
